In [1]:
import warnings
warnings.filterwarnings('ignore')

# Basic libraries
import sys
import numpy as np
import pickle as pk
import plotly.io as pio
import tensorflow as tf
import keras
from tensorflow.keras.models import load_model


pio.renderers.default = "notebook_connected"
print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)


2026-09-16 23:38:00.609359: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Python: /Users/juliowaissman/Documents/proyectos/explaniability-forecasting-with-close-up/.venv/bin/python
TensorFlow: 2.15.1
Keras: 2.15.0


If Keras is 3.X, execute the next cell and restart the kernel. If Keras is 2.X, skip the next cell and continue with the notebook.

In [2]:
# import sys
# !{sys.executable} -m pip uninstall -y tensorflow keras
# !{sys.executable} -m pip install tensorflow==2.15.1 keras==2.15.0


In [3]:
import tree_explainer as te
from exp_model_1 import explain_dict

print(explain_dict["component_names"])

['past-load', 'calendar', 'weather', 'holiday']


In [4]:
# Download the model
path = 'data/calendar_weather_holiday_in_advance/'  
model_name = 'model_calendar_wather_holiday_in_advance.h5'
model = load_model(path + model_name, compile=False)
model.summary()

Model: "ALL"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 encoder_inputs (InputLayer  [(None, 180, 15)]            0         []                            
 )                                                                                                
                                                                                                  
 decoder_inputs (InputLayer  [(None, 36, 14)]             0         []                            
 )                                                                                                
                                                                                                  
 lstm (LSTM)                 [(None, 100),                46400     ['encoder_inputs[0][0]']      
                              (None, 100),                                                      

In [5]:
## Get sample (100 random samples) of the training set to use in the notebook
with open(path + 'X_train_sample.pkl', 'rb') as f:
    X_train_sample = pk.load(f)

In [6]:
# Get some X, e.g. the first element of the test set
with open(path + 'X_encoder_test.pkl', 'rb') as f:
    X_encoder_test = pk.load(f)

with open(path + 'X_decoder_test.pkl', 'rb') as f:
    X_decoder_test = pk.load(f)

X_observation = [X_encoder_test[0], X_decoder_test[0]]

In [7]:
e_mdl = te.Node(explain_dict, model, X_train_sample)

In [8]:
e_mdl.explainability_tree(X_observation)

print("Cumulative execution time:", e_mdl.cum_exec_time)
print("Cumulative model calls:", e_mdl.cum_model_calls)

Cumulative execution time: 26.50398349761963
Cumulative model calls: 70


In [9]:
e_mdl.local_plots()

In [10]:
print(e_mdl)

s, m, e, c_names = e_mdl.unified_explainability()

print("Unified explainability model shape:", m.shape)
print("Unified explainability shapley values:\n", s)
print("Component names:\n", "\n".join(c_names))



te.local_plot([s, m], c_names)

⌙→ self.name
      ⌙---→ past-load
            ⌙---→ past-load 72-180 h before
            ⌙---→ past-load 24-72 h before
            ⌙---→ past-load 1-24 h before
                  ⌙---→ past-load 12-24 h before
                  ⌙---→ past-load 1-12 h before
      ⌙---→ calendar
            ⌙---→ past-calendar 24-180 h before
            ⌙---→ past-calendar 1-24 h before
            ⌙---→ future-calendar
      ⌙---→ weather
            ⌙---→ past-weather
            ⌙---→ future-weather
      ⌙---→ holiday

Unified explainability model shape: (10, 36)
Unified explainability shapley values:
 [-0.29753408 -0.29683176 -0.2981096  -0.3000984  -0.29904813 -0.29456878
 -0.29259664 -0.2938936  -0.298499   -0.303141   -0.30451804 -0.30366287
 -0.30575457 -0.31180796 -0.31579745 -0.31518897 -0.3083461  -0.30171892
 -0.29800165 -0.2976936  -0.30236226 -0.3076659  -0.30846354 -0.3027604
 -0.29667792 -0.2967245  -0.29921916 -0.29986998 -0.29722875 -0.2920844
 -0.2908799  -0.293445   -0.29878613 

In [11]:
e

{'Seq2seq LSTM model': array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.], dtype=float32),
 'past-load': array([0.0228715 , 0.03195766, 0.03939715, 0.04593118, 0.05130656,
        0.05552089, 0.0582843 , 0.05784019, 0.0550586 , 0.05323492,
        0.05145869, 0.04933353, 0.04585743, 0.04243301, 0.03912097,
        0.03570991, 0.0326978 , 0.0307439 , 0.0290203 , 0.02759448,
        0.02907093, 0.03348954, 0.03939058, 0.04490115, 0.04935587,
        0.05367696, 0.05885716, 0.06475996, 0.06991352, 0.0736733 ,
        0.07488975, 0.06857818, 0.06126942, 0.05808984, 0.05650894,
        0.05516711]),
 'past-load 1-24 h before': array([ 0.00414601,  0.00726824,  0.00891268,  0.01017481,  0.011865  ,
         0.01410565,  0.01542844,  0.01500809,  0.01338146,  0.0122972 ,
         0.01125814,  0.01016919,  0.00848528,  0.00666283,  0.00468089,
         0.00260066,  0.00074838, -0